# Whole-image hazard inference
One original RGB image and the same frozen common policy produce visible hazard noun phrases. See `docs/hazard_inference.md`. Select and run one model at a time. Downloads, CUDA loading and inference are separate explicit actions; the default notebook is read-only. CPU tests are synthetic fixtures, and the 6 decimal GB target remains unmeasured.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Optional locations: absolute paths, or paths relative to VLM_evaluation.
DATA_ROOT_OVERRIDE = None
RUN_ROOT_OVERRIDE = None
CACHE_ROOT_OVERRIDE = None
RUN_NAME_OVERRIDE = None

configured_repo = os.environ.get('TRAVERSABILITY_REPO_ROOT')
search_roots = ([Path(configured_repo).expanduser().resolve()] if configured_repo
                else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
candidates = [p for base in search_roots for p in (base, base / 'VLM_evaluation')]
COMPONENT_ROOT = next((p for p in candidates
                       if (p / 'configs/hazards/inference.json').is_file()
                       and (p / 'src/traversability_hazard_inference').is_dir()), None)
if COMPONENT_ROOT is None:
    raise RuntimeError('Set TRAVERSABILITY_REPO_ROOT to VLM_evaluation or its parent checkout.')
source_path = str(COMPONENT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)

from traversability_hazard_inference import load_backend, run_inference
from traversability_hazard_inference.configuration import (
    MODEL_SPECS, component_root, configured_roots, normalize_settings,
)
from traversability_hazard_inference.storage import safe_path

config = json.loads((COMPONENT_ROOT / 'configs/hazards/inference.json').read_text(encoding='utf-8'))

def selected_root(override, default):
    path = Path(override if override is not None else default).expanduser()
    return (path if path.is_absolute() else COMPONENT_ROOT / path).resolve()

roots = configured_roots()
DATA_ROOT = selected_root(DATA_ROOT_OVERRIDE, roots['data_root'])
RUN_ROOT = selected_root(RUN_ROOT_OVERRIDE, roots['run_root'])
CACHE_ROOT = selected_root(CACHE_ROOT_OVERRIDE, roots['cache_root'])
RUN_NAME = config['run_name'] if RUN_NAME_OVERRIDE is None else RUN_NAME_OVERRIDE
RUN_DIR = safe_path(RUN_ROOT, RUN_NAME)
MODEL_KEY = 'qwen3_vl_4b'  # compare qwen3_5_4b next; qwen3_5_2b only by explicit selection
CUDA_DEVICE = 'cuda:0'  # choose one CUDA device explicitly

settings = dict(config['settings'])
settings.update(device=CUDA_DEVICE, cache_dir=str(CACHE_ROOT / 'huggingface/hub'),
                policy_path=str(COMPONENT_ROOT / 'configs/hazards/policy.json'),
                local_files_only=True)
settings = normalize_settings(MODEL_KEY, settings)

# Enable each step explicitly, then rerun only its cell.
DOWNLOAD_WEIGHTS = False
LOAD_MODEL = False
RUN_INFERENCE = False
RETRY_ERRORS = False
if 'backend' not in globals():
    backend = None
if '_backend_model_key' not in globals():
    _backend_model_key = None
if '_backend_settings' not in globals():
    _backend_settings = None

print({'component': str(COMPONENT_ROOT), 'data': str(DATA_ROOT),
       'run': str(RUN_DIR), 'cache': str(CACHE_ROOT), 'model': MODEL_KEY,
       'checkpoint': MODEL_SPECS[MODEL_KEY], 'settings': settings})


## Inspect prepared inputs
The runner needs `frames.jsonl`, dataset identity metadata and the original RGB files under the data root. Prepare the run separately; this notebook does not prepare or download datasets.

In [ ]:
required_inputs = [RUN_DIR / 'frames.jsonl', RUN_DIR / 'metadata/dataset.json']
print({'input_files': {path.relative_to(RUN_DIR).as_posix(): path.is_file()
                       for path in required_inputs}})


## Explicit checkpoint download
Set `DOWNLOAD_WEIGHTS=True` only for a deliberate download of the selected pinned checkpoint. This step uses the configured cache and does not load a model. Skip it for cached/offline weights.

In [ ]:
if DOWNLOAD_WEIGHTS:
    from traversability_hazard_inference.download import download_weights
    cached_checkpoint = download_weights(MODEL_KEY, settings)
    print({'cached_checkpoint': str(cached_checkpoint), 'model': MODEL_KEY})
else:
    print('Download disabled.')


## Explicit model load
Set `LOAD_MODEL=True` and run this cell after preparing the environment and cache. Loading uses cached weights, one selected CUDA device and audited 4-bit language weights. Close an existing backend before loading another model; loading errors never select a fallback automatically.

In [ ]:
if LOAD_MODEL:
    if backend is not None:
        raise RuntimeError('Run the close cell before loading another backend.')
    backend = load_backend(MODEL_KEY, settings)
    _backend_model_key = MODEL_KEY
    _backend_settings = {key: value for key, value in settings.items() if key != 'retry_errors'}
    print({'loaded': True, 'model': MODEL_KEY, 'device': settings['device'],
           'backend_metadata': backend.metadata})
else:
    print('Model loading disabled.')


## Explicit inference and compatible resume
Set `RUN_INFERENCE=True` after loading. The runner publishes metadata before atomic prediction writes and preserves completed records and errors. `RETRY_ERRORS=True` explicitly requests error retries. Changing frozen inputs/settings requires a compatible newly prepared run. The backend remains owned by this notebook until its close step.

In [ ]:
if RUN_INFERENCE:
    if backend is None:
        raise RuntimeError('Run the explicit model-load cell first.')
    current_settings = {key: value for key, value in settings.items() if key != 'retry_errors'}
    if MODEL_KEY != _backend_model_key or current_settings != _backend_settings:
        raise RuntimeError('Model/settings changed: close the backend and load the selected configuration.')
    run_settings = dict(settings, retry_errors=RETRY_ERRORS)
    try:
        summary = run_inference(RUN_DIR, MODEL_KEY, run_settings, DATA_ROOT, backend=backend)
    except BaseException:
        try:
            backend.close()
        finally:
            backend = None
            _backend_model_key = None
            _backend_settings = None
        raise
    print(summary)
else:
    print('Inference disabled.')


## Close this backend
Run after inference and before selecting the second comparison model or the explicit 2B fallback. This releases only the backend owned by this notebook. Repeat the configuration, load and inference steps for the next selected model; do not keep both models resident.

In [ ]:
if backend is not None:
    try:
        backend.close()
    finally:
        backend = None
        _backend_model_key = None
        _backend_settings = None
print('Owned inference backend closed.')
